# Book Page Dewarping — OpenCV walkthrough

This notebook walks through an **artificial demonstration** of the project's boundary detection and piecewise homography. The original academic photograph was not preserved. These synthetic results are software checks, **not an evaluation on real photographed pages**.

In [ ]:
from pathlib import Path
import os
import sys
import cv2
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if not (root / 'dewarp').exists():
    root = root.parent
if not (root / 'dewarp').exists():
    raise RuntimeError('Run this notebook from the repository or notebooks/ directory')
sys.path.insert(0, str(root))
from scripts.generate_demo import generate
from dewarp import detect_boundaries, piecewise_homography, shading_correction
from dewarp.boundaries import overlay_points
source = generate(root / 'data/demo/synthetic_curved_page.png')
image = cv2.imread(str(source))
print('Image:', image.shape, '| synthetic demonstration')

## 1. Inspect source and boundaries

The detector uses a contrasting page region when available, and falls back to Canny edge scanning on less distinct backgrounds. The original study relied on manually refining 35 upper/lower guide-point pairs.

In [ ]:
bounds = detect_boundaries(image, num_points=35)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
ax[1].imshow(cv2.cvtColor(overlay_points(image,bounds), cv2.COLOR_BGR2RGB))
for a,t in zip(ax, ['Synthetic image', 'Detected control points']):
    a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

## 2. Local piecewise homography

Each adjacent control-point pair defines a strip. The page is further divided into horizontal cells; each cell receives its own perspective transform. This version warps one small output patch at a time rather than repeating a full-image transform for every cell.

In [ ]:
warped = piecewise_homography(image, bounds, grid_rows=30)
corrected = shading_correction(warped)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(cv2.cvtColor(warped,cv2.COLOR_BGR2RGB))
ax[1].imshow(corrected, cmap='gray', vmin=0, vmax=255)
for a,t in zip(ax, ['Warped image', 'Shading-corrected grayscale']):
    a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

## 3. Working with real photographs

Use `python -m dewarp.cli --input data/raw/your_page.jpg --overlay outputs/points.png --interactive` from the repository root for editable guide points. The interactive mode requires a desktop environment. The archived academic notebook and report describe the original project; the transformed output cannot be reproduced without the missing input photograph.

**Limitations:** Boundary detection depends on visible page edges. Local homographies can leave seams when control points are inaccurate. No accuracy or OCR improvement metrics are claimed.